In [ ]:
import { useState, useMemo, useCallback, useRef } from "react";
import * as XLSX from "xlsx";

/* ───────────────────── HELPERS ───────────────────── */
const fmt = (n) => new Intl.NumberFormat("es-AR", { style: "currency", currency: "ARS", minimumFractionDigits: 0 }).format(n);
const pct = (n) => (n >= 0 ? "+" : "") + n.toFixed(1) + "%";
const fmtDate = (d) => { if (!d) return ""; if (typeof d === "number") { try { const dt = XLSX.SSF.parse_date_code(d); return `${String(dt.d).padStart(2,"0")}/${String(dt.m).padStart(2,"0")}/${dt.y}`; } catch(e) { return d.toString(); } } return d.toString(); };

/* ───────────────── INFLACIÓN ARGENTINA 2025 (INDEC) ───────────────── */
const INFLATION_DATA = [
  { month: "Ene 25", rate: 2.2 }, { month: "Feb 25", rate: 2.4 }, { month: "Mar 25", rate: 3.7 },
  { month: "Abr 25", rate: 2.8 }, { month: "May 25", rate: 1.5 }, { month: "Jun 25", rate: 1.6 },
  { month: "Jul 25", rate: 1.7 }, { month: "Ago 25", rate: 1.9 }, { month: "Sep 25", rate: 2.1 },
  { month: "Oct 25", rate: 2.3 }, { month: "Nov 25", rate: 2.4 }, { month: "Dic 25", rate: 2.8 },
  { month: "Ene 26*", rate: 2.5 },
];
const INFLATION_ANNUAL_2025 = 31.5;
const INFLATION_PROJECTED_2026 = 20.1;
const AVG_MONTHLY = INFLATION_DATA.reduce((s, d) => s + d.rate, 0) / INFLATION_DATA.length;

/* ───────────────── EXCEL PARSER (optimized for large files) ───────────────── */
function normalizeRow(row) {
  const get = (...keys) => { 
    for (const k of keys) { 
      const v = row[k]; 
      if (v !== undefined && v !== null && v !== "") return v; 
    } 
    return null; 
  };
  
  // Normalización de columnas comunes
  const date = get("F.Ingreso", "f.ingreso", "F. Ingreso", "Fecha", "fecha", "FIngreso");
  const desc = get("Descripción artículo", "Descripcion articulo", "Articulo", "articulo", "Producto", "producto", "DESCRIPCION", "Descripción");
  const qty = get("Cantidad", "cantidad", "Cant", "cant", "Qty");
  const price = get("Precio", "precio", " Precio ", "Precio ", "Costo", "costo");
  const amount = get("Monto", "monto", " Monto ", "Monto ", "Total", "total");
  const cat = get("Categoría", "Categoria", "categoria", "Rubro", "rubro");
  
  // Detección de columna TIPO (clave para tu caso)
  const type = get("Tipo", "TIPO", "tipo", "Movimiento", "movimiento");

  return {
    date: date ? fmtDate(date) : "",
    description: desc ? desc.toString().trim() : "",
    qty: Number(qty) || 0,
    price: Number(String(price).replace(/[^0-9.,\-]/g, "").replace(",", ".")) || 0,
    amount: Number(String(amount).replace(/[^0-9.,\-]/g, "").replace(",", ".")) || 0,
    category: cat ? cat.toString().trim() : "General",
    type: type ? type.toString().trim().toUpperCase() : null // Normalizamos a mayúsculas
  };
}

function parseWorkbook(workbook, onProgress) {
  const sheets = workbook.SheetNames;
  
  // Estrategia: Leemos la primera hoja para ver qué formato tiene
  if (onProgress) onProgress("Analizando estructura del archivo...");
  const firstSheetName = sheets[0];
  const rawData = XLSX.utils.sheet_to_json(workbook.Sheets[firstSheetName]);

  // 1. Detección MODO TIPO: ¿Hay una columna "Tipo" en la primera hoja?
  // Esto soluciona tu problema actual.
  const hasTipoColumn = rawData.length > 0 && normalizeRow(rawData[0]).type !== null;

  if (hasTipoColumn) {
    if (onProgress) onProgress("Detectado formato unificado con columna TIPO...");
    
    const purchaseMap = {};
    const salesMap = {};

    rawData.forEach(row => {
      const r = normalizeRow(row);
      if (!r.description) return;

      // Lógica de decisión: ¿Es Compra o Venta?
      const isPurchase = r.type.includes("COMPRA") || r.type.includes("INGRESO") || r.type.includes("ENTRADA");
      const isSale = r.type.includes("VENTA") || r.type.includes("EGRESO") || r.type.includes("SALIDA");

      // Seleccionamos el mapa correcto
      let map = null;
      if (isPurchase) map = purchaseMap;
      else if (isSale) map = salesMap;
      
      if (map) {
        if (!map[r.description]) {
          map[r.description] = { qty: 0, totalAmount: 0, prices: [], category: r.category, rows: [] };
        }
        const m = map[r.description];
        
        // Acumulamos
        m.qty += r.qty;
        // Si no hay monto, lo calculamos (precio * cantidad)
        const rowTotal = r.amount || (r.qty * r.price);
        m.totalAmount += rowTotal;
        
        if (r.price > 0) m.prices.push(r.price);
        
        // Guardamos historial (limitado para optimizar memoria)
        if (m.rows.length < 50) m.rows.push(r);
      }
    });

    if (onProgress) onProgress("Procesando artículos...");
    return buildProductsFromMaps(purchaseMap, salesMap, rawData.length, rawData.length, "Unificado (TIPO)", "-");
  }

  // 2. MODO RESUMEN (Python script anterior que generaba columnas "cant_comprada", etc.)
  const colKeys = rawData.length > 0 ? Object.keys(rawData[0]).map(k=>k.toLowerCase()) : [];
  const isResumen = colKeys.some(k => k.includes("cant_vendida") || k.includes("monto_venta"));

  if (isResumen) {
    if (onProgress) onProgress("Detectado formato Resumen consolidado...");
    // ... (Lógica para leer el resumen generado por Python si usas el script anterior)
    // Para simplificar, si usas el nuevo script con TIPO, entrará en el IF de arriba.
    // Si necesitas este bloque también, avísame y te lo paso completo.
  }

  // 3. MODO CLÁSICO (Hoja 1 = Compras, Hoja 2 = Ventas)
  // Este es el fallback si no hay columna TIPO.
  if (sheets.length >= 2) {
    if (onProgress) onProgress("Formato estándar: Hoja 1 Compras / Hoja 2 Ventas...");
    
    const rawP = XLSX.utils.sheet_to_json(workbook.Sheets[sheets[0]]);
    const rawS = XLSX.utils.sheet_to_json(workbook.Sheets[sheets[1]]);
    
    const purchaseMap = {};
    const salesMap = {};

    // Procesar Compras (Hoja 1)
    rawP.forEach(row => {
      const r = normalizeRow(row);
      if (!r.description) return;
      if (!purchaseMap[r.description]) purchaseMap[r.description] = { qty: 0, totalAmount: 0, prices: [], category: r.category, rows: [] };
      const m = purchaseMap[r.description];
      m.qty += r.qty;
      m.totalAmount += r.amount || (r.qty * r.price);
      if (r.price > 0) m.prices.push(r.price);
      if (m.rows.length < 50) m.rows.push(r);
    });

    // Procesar Ventas (Hoja 2)
    rawS.forEach(row => {
      const r = normalizeRow(row);
      if (!r.description) return;
      if (!salesMap[r.description]) salesMap[r.description] = { qty: 0, totalAmount: 0, prices: [], category: r.category, rows: [] };
      const m = salesMap[r.description];
      m.qty += r.qty;
      m.totalAmount += r.amount || (r.qty * r.price);
      if (r.price > 0) m.prices.push(r.price);
      if (m.rows.length < 50) m.rows.push(r);
    });

    return buildProductsFromMaps(purchaseMap, salesMap, rawP.length, rawS.length, sheets[0], sheets[1]);
  }

  return { error: "Formato no reconocido. Asegúrate de tener una columna 'TIPO' o usar 2 hojas separadas." };
}

function buildProductsFromMaps(purchaseMap, salesMap, pCount, sCount, s1, s2) {
  const allArticles = new Set([...Object.keys(purchaseMap), ...Object.keys(salesMap)]);
  const products = []; const discrepancies = [];
  allArticles.forEach(name => {
    const p = purchaseMap[name] || { qty: 0, totalAmount: 0, prices: [], dates: [], category: "General", rows: [] };
    const s = salesMap[name] || { qty: 0, totalAmount: 0, prices: [], dates: [], category: "General", rows: [] };
    const avgCost = p.prices.length > 0 ? p.prices.reduce((a, b) => a + b, 0) / p.prices.length : 0;
    const avgPrice = s.prices.length > 0 ? s.prices.reduce((a, b) => a + b, 0) / s.prices.length : 0;
    const costHistory = []; let lastC = null;
    for (const r of (p.rows||[])) { if (r.price > 0 && r.price !== lastC) { costHistory.push({ date: r.date||"", cost: r.price }); lastC = r.price; } }
    const priceHistory = []; let lastP = null;
    for (const r of (s.rows||[])) { if (r.price > 0 && r.price !== lastP) { priceHistory.push({ date: r.date||"", price: r.price }); lastP = r.price; } }
    const onlyP = s.qty === 0 && p.qty > 0; const onlyS = p.qty === 0 && s.qty > 0;
    products.push({ id: Date.now()+Math.random(), name, category: p.category||s.category||"General", qtyBought: p.qty, qtySold: s.qty, totalCostAmount: p.totalAmount, totalSaleAmount: s.totalAmount, avgCost, avgPrice, costHistory: costHistory.length>0?costHistory:[{date:"",cost:avgCost}], priceHistory: priceHistory.length>0?priceHistory:[{date:"",price:avgPrice}], purchaseRows: p.rows||[], saleRows: s.rows||[], onlyPurchase: onlyP, onlySale: onlyS });
    if (onlyP) discrepancies.push({ article: name, type: "solo_compra", purchaseQty: p.qty, saleQty: 0 });
    if (onlyS) discrepancies.push({ article: name, type: "solo_venta", purchaseQty: 0, saleQty: s.qty });
  });
  return { products, discrepancies, purchaseCount: pCount||0, salesCount: sCount||0, sheet1Name: s1||"Compras", sheet2Name: s2||"Ventas" };
}

/* ───────────── TEMPLATE / EXPORT ───────────── */
function downloadTemplate() {
  const pd = [
    { "F.Ingreso": "06/01/2025", "Descripción artículo": "Leche Entera 1L", Cantidad: 200, " Precio ": 850, " Monto ": 170000, "Descripción": "Lácteos" },
    { "F.Ingreso": "06/01/2025", "Descripción artículo": "Pan Lactal", Cantidad: 300, " Precio ": 1200, " Monto ": 360000, "Descripción": "Panadería" },
    { "F.Ingreso": "13/01/2025", "Descripción artículo": "Leche Entera 1L", Cantidad: 150, " Precio ": 920, " Monto ": 138000, "Descripción": "Lácteos" },
    { "F.Ingreso": "13/01/2025", "Descripción artículo": "Coca-Cola 2.25L", Cantidad: 100, " Precio ": 1800, " Monto ": 180000, "Descripción": "Bebidas" },
  ];
  const sd = [
    { "F.Ingreso": "06/01/2025", "Descripción artículo": "Leche Entera 1L", Cantidad: 180, " Precio ": 1200, " Monto ": 216000, "Descripción": "Lácteos" },
    { "F.Ingreso": "06/01/2025", "Descripción artículo": "Pan Lactal", Cantidad: 280, " Precio ": 1800, " Monto ": 504000, "Descripción": "Panadería" },
    { "F.Ingreso": "13/01/2025", "Descripción artículo": "Yerba Mate 1kg", Cantidad: 50, " Precio ": 3500, " Monto ": 175000, "Descripción": "Almacén" },
  ];
  const wb = XLSX.utils.book_new();
  const w1 = XLSX.utils.json_to_sheet(pd); w1["!cols"] = [{wch:14},{wch:24},{wch:10},{wch:12},{wch:14},{wch:14}];
  XLSX.utils.book_append_sheet(wb, w1, "Compras");
  const w2 = XLSX.utils.json_to_sheet(sd); w2["!cols"] = [{wch:14},{wch:24},{wch:10},{wch:12},{wch:14},{wch:14}];
  XLSX.utils.book_append_sheet(wb, w2, "Ventas");
  XLSX.writeFile(wb, "plantilla_superanalytics.xlsx");
}

function exportData(products) {
  const wb = XLSX.utils.book_new();
  const pR = [], sR = [];
  products.forEach(p => {
    p.purchaseRows.forEach(r => pR.push({ "F.Ingreso": r.date, "Descripción artículo": p.name, Cantidad: r.qty, " Precio ": r.price, " Monto ": r.amount, "Descripción": p.category }));
    p.saleRows.forEach(r => sR.push({ "F.Ingreso": r.date, "Descripción artículo": p.name, Cantidad: r.qty, " Precio ": r.price, " Monto ": r.amount, "Descripción": p.category }));
  });
  XLSX.utils.book_append_sheet(wb, XLSX.utils.json_to_sheet(pR), "Compras");
  XLSX.utils.book_append_sheet(wb, XLSX.utils.json_to_sheet(sR), "Ventas");
  XLSX.writeFile(wb, "superanalytics_export.xlsx");
}

/* ───────────────── UI COMPONENTS ───────────────── */
function MiniBar({ value, max, color = "#4ade80", height = 16 }) {
  const w = max > 0 ? (value / max) * 100 : 0;
  return (<div style={{ background: "#1a1a2e", borderRadius: 4, height, width: "100%", overflow: "hidden" }}><div style={{ background: color, height: "100%", width: `${Math.min(w,100)}%`, borderRadius: 4, transition: "width .5s ease" }} /></div>);
}

function KPIChanges({ product }) {
  const ch = [];
  for (let i = 1; i < product.costHistory.length; i++) { const p = product.costHistory[i-1], c = product.costHistory[i]; ch.push({ t:"COSTO", d:c.date, f:p.cost, to:c.cost, ch:((c.cost-p.cost)/p.cost)*100 }); }
  for (let i = 1; i < product.priceHistory.length; i++) { const p = product.priceHistory[i-1], c = product.priceHistory[i]; ch.push({ t:"PRECIO", d:c.date, f:p.price, to:c.price, ch:((c.price-p.price)/p.price)*100 }); }
  if (!ch.length) return <div style={{ color: "#64748b", fontSize: 12, padding: "6px 0" }}>Sin cambios registrados</div>;
  return (<div style={{ display: "flex", flexDirection: "column", gap: 5 }}>{ch.map((c,i) => (
    <div key={i} style={{ display: "flex", alignItems: "center", gap: 6, fontSize: 11, flexWrap: "wrap" }}>
      <span style={{ background: c.t==="COSTO"?"#dc2626":"#2563eb", color: "#fff", borderRadius: 4, padding: "1px 5px", fontWeight: 700, fontSize: 9 }}>{c.t}</span>
      <span style={{ color: "#94a3b8" }}>{c.d}</span><span style={{ color: c.t==="COSTO"?"#f87171":"#60a5fa" }}>{fmt(c.f)}</span><span style={{ color: "#475569" }}>→</span>
      <span style={{ color: "#fbbf24" }}>{fmt(c.to)}</span><span style={{ color: c.ch>0?"#f87171":"#4ade80", fontWeight: 700 }}>{pct(c.ch)}</span>
    </div>))}</div>);
}

function InflationBanner() {
  const [open, setOpen] = useState(false);
  const latest = INFLATION_DATA[INFLATION_DATA.length-1];
  const maxR = Math.max(...INFLATION_DATA.map(d=>d.rate));
  return (
    <div style={{ background: "linear-gradient(135deg,#1e1b4b,#0f172a)", border: "1px solid #312e81", borderRadius: 14, padding: "14px 18px", marginBottom: 14 }}>
      <div style={{ display: "flex", justifyContent: "space-between", alignItems: "center", cursor: "pointer", flexWrap: "wrap", gap: 8 }} onClick={()=>setOpen(!open)}>
        <div style={{ display: "flex", alignItems: "center", gap: 8 }}><span style={{ fontSize: 20 }}>📉</span><div><div style={{ fontSize: 12, fontWeight: 700, color: "#c4b5fd" }}>Inflación Argentina (INDEC)</div><div style={{ fontSize: 10, color: "#7c3aed" }}>Último: {latest.month}</div></div></div>
        <div style={{ display: "flex", gap: 14, alignItems: "center" }}>
          <div style={{ textAlign: "right" }}><div style={{ fontSize: 9, color: "#64748b" }}>MES</div><div style={{ fontFamily: "'JetBrains Mono',monospace", fontSize: 16, fontWeight: 800, color: "#fbbf24" }}>{latest.rate}%</div></div>
          <div style={{ textAlign: "right" }}><div style={{ fontSize: 9, color: "#64748b" }}>2025</div><div style={{ fontFamily: "'JetBrains Mono',monospace", fontSize: 16, fontWeight: 800, color: "#f87171" }}>{INFLATION_ANNUAL_2025}%</div></div>
          <div style={{ textAlign: "right" }}><div style={{ fontSize: 9, color: "#64748b" }}>PROY 26</div><div style={{ fontFamily: "'JetBrains Mono',monospace", fontSize: 16, fontWeight: 800, color: "#4ade80" }}>{INFLATION_PROJECTED_2026}%</div></div>
          <span style={{ color: "#64748b", fontSize: 16, transform: open?"rotate(180deg)":"rotate(0)", transition: ".2s" }}>▾</span>
        </div>
      </div>
      {open && (<div style={{ marginTop: 14, paddingTop: 14, borderTop: "1px solid #1e293b" }}>
        <div style={{ display: "flex", alignItems: "flex-end", gap: 3, height: 90 }}>
          {INFLATION_DATA.map((d,i) => (<div key={i} style={{ flex: 1, display: "flex", flexDirection: "column", alignItems: "center", gap: 3 }}>
            <span style={{ fontSize: 8, fontFamily: "'JetBrains Mono',monospace", color: "#c4b5fd", fontWeight: 600 }}>{d.rate}%</span>
            <div style={{ width: "100%", maxWidth: 28, height: `${(d.rate/maxR)*60}px`, background: d.rate>=2.5?"#f87171":d.rate>=2?"#fbbf24":"#4ade80", borderRadius: "3px 3px 0 0" }} />
            <span style={{ fontSize: 7, color: "#64748b" }}>{d.month}</span>
          </div>))}
        </div>
      </div>)}
    </div>
  );
}

function InflationEstimator({ products, page, perPage }) {
  const visible = products.slice((page-1)*perPage, page*perPage);
  const estimates = useMemo(() => visible.filter(p => p.avgCost>0||p.avgPrice>0).map(p => {
    const r = AVG_MONTHLY/100;
    return { ...p, estCost: Math.round(p.avgCost*(1+r)), estPrice: Math.round(p.avgPrice*(1+r)) };
  }), [visible]);
  return (
    <div style={{ background: "linear-gradient(135deg,#0f172a,#1e293b)", border: "1px solid #1e293b", borderRadius: 14, padding: 18 }}>
      <h3 style={{ margin: "0 0 4px", fontSize: 13, fontWeight: 700, color: "#c4b5fd" }}>🔮 Estimación por Inflación (prom. mensual {AVG_MONTHLY.toFixed(1)}%)</h3>
      <div style={{ display: "flex", flexDirection: "column", gap: 6, marginTop: 10 }}>
        {estimates.map(e => (
          <div key={e.id} style={{ display: "grid", gridTemplateColumns: "1fr auto auto auto", gap: 10, alignItems: "center", padding: "8px 12px", borderRadius: 8, background: "rgba(15,23,42,.5)", border: e.onlySale?"1px solid rgba(251,191,36,.2)":"1px solid transparent" }}>
            <div><div style={{ fontSize: 12, fontWeight: 600 }}>{e.name}{e.onlySale&&<span style={{ marginLeft:6,fontSize:9,color:"#fbbf24",fontWeight:700 }}>SIN COMPRA</span>}</div><div style={{ fontSize: 10, color: "#64748b" }}>{e.category}</div></div>
            <div style={{ textAlign: "right" }}><div style={{ fontSize: 9, color: "#64748b" }}>Costo → est.</div><div style={{ fontSize: 11, fontFamily: "'JetBrains Mono',monospace" }}><span style={{ color: "#f87171" }}>{fmt(e.avgCost)}</span> → <span style={{ color: "#fbbf24", fontWeight: 700 }}>{fmt(e.estCost)}</span></div></div>
            <div style={{ textAlign: "right" }}><div style={{ fontSize: 9, color: "#64748b" }}>Precio → est.</div><div style={{ fontSize: 11, fontFamily: "'JetBrains Mono',monospace" }}><span style={{ color: "#4ade80" }}>{fmt(e.avgPrice)}</span> → <span style={{ color: "#fbbf24", fontWeight: 700 }}>{fmt(e.estPrice)}</span></div></div>
            <div style={{ textAlign: "right" }}><div style={{ fontSize: 9, color: "#64748b" }}>Margen</div><div style={{ fontSize: 12, fontFamily: "'JetBrains Mono',monospace", fontWeight: 700, color: "#a78bfa" }}>{e.estPrice>0?((e.estPrice-e.estCost)/e.estPrice*100).toFixed(1):0}%</div></div>
          </div>
        ))}
      </div>
    </div>
  );
}

function DiscrepancyTable({ discrepancies }) {
  const sc = discrepancies.filter(d=>d.type==="solo_compra");
  const sv = discrepancies.filter(d=>d.type==="solo_venta");
  if (!discrepancies.length) return (<div style={{ background: "rgba(74,222,128,.08)", border: "1px solid rgba(74,222,128,.3)", borderRadius: 12, padding: 16, textAlign: "center" }}><span style={{ fontSize: 20 }}>✅</span><div style={{ fontSize: 14, fontWeight: 700, color: "#4ade80", marginTop: 6 }}>Sin discrepancias</div><div style={{ fontSize: 12, color: "#64748b", marginTop: 4 }}>Todos los artículos de compra tienen registro de venta y viceversa.</div></div>);
  return (
    <div style={{ display: "flex", flexDirection: "column", gap: 12 }}>
      {sc.length>0&&(<div style={{ background: "rgba(251,146,60,.06)", border: "1px solid rgba(251,146,60,.25)", borderRadius: 12, padding: 16 }}>
        <div style={{ display: "flex", alignItems: "center", gap: 8, marginBottom: 10 }}><span style={{ fontSize: 16 }}>🟠</span><div><div style={{ fontSize: 13, fontWeight: 700, color: "#fb923c" }}>Se compraron pero NO se vendieron ({sc.length})</div></div></div>
        <div style={{ display: "flex", flexDirection: "column", gap: 4 }}>{sc.map((d,i)=>(<div key={i} style={{ display: "flex", justifyContent: "space-between", padding: "8px 12px", background: "rgba(251,146,60,.05)", borderRadius: 8, fontSize: 12 }}><span style={{ fontWeight: 600 }}>{d.article}</span><span style={{ fontFamily: "'JetBrains Mono',monospace", color: "#fb923c", fontWeight: 600 }}>Comprado: {d.purchaseQty} u.</span></div>))}</div>
      </div>)}
      {sv.length>0&&(<div style={{ background: "rgba(239,68,68,.06)", border: "1px solid rgba(239,68,68,.25)", borderRadius: 12, padding: 16 }}>
        <div style={{ display: "flex", alignItems: "center", gap: 8, marginBottom: 10 }}><span style={{ fontSize: 16 }}>🔴</span><div><div style={{ fontSize: 13, fontWeight: 700, color: "#ef4444" }}>Se vendieron pero NO se compraron ({sv.length})</div></div></div>
        <div style={{ display: "flex", flexDirection: "column", gap: 4 }}>{sv.map((d,i)=>(<div key={i} style={{ display: "flex", justifyContent: "space-between", padding: "8px 12px", background: "rgba(239,68,68,.05)", borderRadius: 8, fontSize: 12 }}><span style={{ fontWeight: 600 }}>{d.article}</span><span style={{ fontFamily: "'JetBrains Mono',monospace", color: "#ef4444", fontWeight: 600 }}>Vendido: {d.saleQty} u.</span></div>))}</div>
      </div>)}
    </div>
  );
}

/* ───────── PAGINATION ───────── */
function Pagination({ total, page, perPage, onChange }) {
  const pages = Math.ceil(total / perPage);
  if (pages <= 1) return null;
  const start = (page-1)*perPage+1;
  const end = Math.min(page*perPage, total);
  return (
    <div style={{ display: "flex", justifyContent: "space-between", alignItems: "center", padding: "10px 0", flexWrap: "wrap", gap: 8 }}>
      <span style={{ fontSize: 11, color: "#64748b" }}>Mostrando {start}-{end} de {total.toLocaleString()} artículos</span>
      <div style={{ display: "flex", gap: 4 }}>
        <button onClick={()=>onChange(1)} disabled={page===1} style={{ padding: "4px 10px", borderRadius: 5, border: "1px solid #334155", background: "transparent", color: page===1?"#334155":"#94a3b8", cursor: page===1?"default":"pointer", fontSize: 11 }}>«</button>
        <button onClick={()=>onChange(page-1)} disabled={page===1} style={{ padding: "4px 10px", borderRadius: 5, border: "1px solid #334155", background: "transparent", color: page===1?"#334155":"#94a3b8", cursor: page===1?"default":"pointer", fontSize: 11 }}>‹</button>
        <span style={{ padding: "4px 12px", fontSize: 11, color: "#e2e8f0", fontWeight: 600 }}>{page} / {pages}</span>
        <button onClick={()=>onChange(page+1)} disabled={page===pages} style={{ padding: "4px 10px", borderRadius: 5, border: "1px solid #334155", background: "transparent", color: page===pages?"#334155":"#94a3b8", cursor: page===pages?"default":"pointer", fontSize: 11 }}>›</button>
        <button onClick={()=>onChange(pages)} disabled={page===pages} style={{ padding: "4px 10px", borderRadius: 5, border: "1px solid #334155", background: "transparent", color: page===pages?"#334155":"#94a3b8", cursor: page===pages?"default":"pointer", fontSize: 11 }}>»</button>
      </div>
    </div>
  );
}

/* ───────── IMPORT MODAL ───────── */
function ImportPanel({ onImport, onClose, count }) {
  const fileRef = useRef(null);
  const [status, setStatus] = useState(null);
  const [mode, setMode] = useState("replace");
  const [loading, setLoading] = useState(false);
  const [progress, setProgress] = useState("");

  const handleFile = (e) => {
    const file = e.target.files[0]; if (!file) return;
    setStatus(null); setLoading(true);
    const sizeMB = (file.size / 1024 / 1024).toFixed(1);
    setProgress(`Cargando archivo (${sizeMB} MB)...`);

    const reader = new FileReader();
    reader.onload = (evt) => {
      try {
        setProgress("Parseando Excel...");
        const wb = XLSX.read(new Uint8Array(evt.target.result), { type: "array" });
        const result = parseWorkbook(wb, (msg) => setProgress(msg));
        if (result.error) { setStatus({ type: "error", msg: result.error }); setLoading(false); return; }
        setProgress("Renderizando...");
        // Use setTimeout to let the UI update before heavy state change
        setTimeout(() => {
          onImport(result, mode);
          setStatus({ type: "success", msg: `✅ ${result.products.length.toLocaleString()} artículos (${result.purchaseCount.toLocaleString()} compras + ${result.salesCount.toLocaleString()} ventas). ${result.discrepancies.length} discrepancias.` });
          setLoading(false); setProgress("");
        }, 50);
      } catch (err) { setStatus({ type: "error", msg: "Error: " + err.message }); setLoading(false); }
    };
    reader.readAsArrayBuffer(file);
  };

  return (
    <div style={{ position: "fixed", inset: 0, background: "rgba(0,0,0,.75)", display: "flex", alignItems: "center", justifyContent: "center", zIndex: 999 }} onClick={onClose}>
      <div onClick={e=>e.stopPropagation()} style={{ background: "#0f172a", border: "1px solid #1e293b", borderRadius: 18, padding: 28, width: 540, maxWidth: "95vw", display: "flex", flexDirection: "column", gap: 16 }}>
        <div style={{ display: "flex", justifyContent: "space-between" }}><h3 style={{ margin: 0, color: "#e2e8f0", fontSize: 18, fontWeight: 700 }}>📥 Importar Excel</h3><button onClick={onClose} style={{ background: "none", border: "none", color: "#64748b", fontSize: 20, cursor: "pointer" }}>✕</button></div>

        <div style={{ background: "#1e293b", borderRadius: 10, padding: 14 }}>
          <div style={{ fontSize: 12, fontWeight: 600, color: "#94a3b8", marginBottom: 8 }}>Estructura:</div>
          <div style={{ display: "flex", gap: 16, marginBottom: 8 }}>
            <div><span style={{ fontSize: 11, color: "#4ade80", fontWeight: 700 }}>Hoja 1</span><span style={{ fontSize: 11, color: "#64748b" }}> → Compras</span></div>
            <div><span style={{ fontSize: 11, color: "#60a5fa", fontWeight: 700 }}>Hoja 2</span><span style={{ fontSize: 11, color: "#64748b" }}> → Ventas</span></div>
          </div>
          <div style={{ fontSize: 11, color: "#64748b", marginBottom: 4 }}>Columnas:</div>
          <div style={{ display: "flex", flexWrap: "wrap", gap: "3px 12px" }}>
            {["F.Ingreso","Descripción artículo","Cantidad","Precio","Monto","Descripción"].map(c=>(<span key={c} style={{ fontFamily: "'JetBrains Mono',monospace", color: "#60a5fa", fontWeight: 600, fontSize: 11 }}>{c}</span>))}
          </div>
          <div style={{ marginTop: 8, fontSize: 10, color: "#475569", background: "rgba(74,222,128,.05)", padding: "6px 10px", borderRadius: 6, border: "1px solid rgba(74,222,128,.15)" }}>
            💡 Soporta archivos grandes: +100.000 filas sin problemas
          </div>
        </div>

        <button onClick={downloadTemplate} style={{ padding: "10px", background: "linear-gradient(135deg,#1e3a5f,#1e293b)", border: "1px solid #334155", borderRadius: 8, color: "#38bdf8", cursor: "pointer", fontSize: 12, fontWeight: 600 }}>📋 Descargar Plantilla</button>

        <div style={{ display: "flex", gap: 10 }}>
          <span style={{ color: "#64748b", fontSize: 11, fontWeight: 600, lineHeight: "32px" }}>MODO:</span>
          {[{v:"replace",l:"Reemplazar"},{v:"append",l:"Agregar"}].map(m=>(<button key={m.v} onClick={()=>setMode(m.v)} style={{ padding: "6px 14px", borderRadius: 6, border: `1px solid ${mode===m.v?"#2563eb":"#334155"}`, background: mode===m.v?"rgba(37,99,235,.15)":"transparent", color: mode===m.v?"#60a5fa":"#94a3b8", cursor: "pointer", fontSize: 11, fontWeight: 600 }}>{m.l}</button>))}
        </div>

        <input ref={fileRef} type="file" accept=".xlsx,.xls,.csv" onChange={handleFile} style={{ display: "none" }} />
        <button onClick={()=>fileRef.current?.click()} disabled={loading} style={{ padding: "12px", borderRadius: 10, border: "2px dashed #334155", background: loading?"rgba(37,99,235,.1)":"rgba(30,41,59,.5)", color: "#e2e8f0", cursor: loading?"wait":"pointer", fontSize: 13, fontWeight: 600 }}>
          {loading ? "⏳ Procesando..." : "📂 Seleccionar archivo (.xlsx, .xls)"}
        </button>

        {loading && progress && (
          <div style={{ padding: "10px 14px", borderRadius: 8, background: "rgba(37,99,235,.08)", border: "1px solid rgba(37,99,235,.3)" }}>
            <div style={{ display: "flex", alignItems: "center", gap: 8 }}>
              <div style={{ width: 14, height: 14, border: "2px solid #2563eb", borderTop: "2px solid transparent", borderRadius: "50%", animation: "spin 1s linear infinite" }} />
              <span style={{ fontSize: 12, color: "#60a5fa", fontWeight: 600 }}>{progress}</span>
            </div>
          </div>
        )}

        {status && <div style={{ padding: "10px 14px", borderRadius: 8, fontSize: 12, fontWeight: 600, background: status.type==="success"?"rgba(74,222,128,.08)":"rgba(248,113,113,.08)", border: `1px solid ${status.type==="success"?"#4ade80":"#f87171"}`, color: status.type==="success"?"#4ade80":"#f87171" }}>{status.msg}</div>}
        {count>0&&<div style={{ fontSize: 10, color: "#64748b", textAlign: "center" }}>{count} artículos cargados actualmente</div>}
      </div>
      <style>{`@keyframes spin { to { transform: rotate(360deg); } }`}</style>
    </div>
  );
}

/* ───────────────── SEARCH ───────────────── */
function SearchBar({ value, onChange }) {
  return (
    <input value={value} onChange={e=>onChange(e.target.value)} placeholder="🔍 Buscar artículo..."
      style={{ padding: "7px 12px", borderRadius: 7, background: "#1e293b", border: "1px solid #334155", color: "#e2e8f0", fontSize: 12, width: 200, outline: "none" }} />
  );
}

/* ───────────────── MAIN APP ───────────────── */
const PER_PAGE = 50;

export default function SupermarketAnalytics() {
  const [products, setProducts] = useState([]);
  const [discrepancies, setDiscrepancies] = useState([]);
  const [selectedProduct, setSelectedProduct] = useState(null);
  const [view, setView] = useState("dashboard");
  const [showImport, setShowImport] = useState(false);
  const [filterCategory, setFilterCategory] = useState("all");
  const [search, setSearch] = useState("");
  const [page, setPage] = useState(1);
  const [sortBy, setSortBy] = useState("name"); // name, margin, profit, bought, sold
  const [sortDir, setSortDir] = useState(1);

  const categories = useMemo(() => ["all", ...new Set(products.map(p=>p.category))], [products]);

  const filtered = useMemo(() => {
    let list = filterCategory === "all" ? products : products.filter(p=>p.category===filterCategory);
    if (search.trim()) { const s = search.toLowerCase(); list = list.filter(p=>p.name.toLowerCase().includes(s)); }
    list.sort((a,b) => {
      let va, vb;
      if (sortBy==="name") { va=a.name.toLowerCase(); vb=b.name.toLowerCase(); return va<vb?-sortDir:va>vb?sortDir:0; }
      if (sortBy==="margin") { va=a.avgPrice>0?(a.avgPrice-a.avgCost)/a.avgPrice:0; vb=b.avgPrice>0?(b.avgPrice-b.avgCost)/b.avgPrice:0; }
      else if (sortBy==="profit") { va=a.totalSaleAmount-a.totalCostAmount; vb=b.totalSaleAmount-b.totalCostAmount; }
      else if (sortBy==="bought") { va=a.qtyBought; vb=b.qtyBought; }
      else if (sortBy==="sold") { va=a.qtySold; vb=b.qtySold; }
      else { va=0; vb=0; }
      return (va-vb)*sortDir;
    });
    return list;
  }, [products, filterCategory, search, sortBy, sortDir]);

  const pageProducts = useMemo(() => filtered.slice((page-1)*PER_PAGE, page*PER_PAGE), [filtered, page]);

  const stats = useMemo(() => {
    let rev=0, cost=0, bought=0, sold=0;
    filtered.forEach(p => { rev+=p.totalSaleAmount; cost+=p.totalCostAmount; bought+=p.qtyBought; sold+=p.qtySold; });
    return { rev, cost, profit: rev-cost, margin: rev>0?((rev-cost)/rev)*100:0, bought, sold };
  }, [filtered]);

  const handleImport = useCallback((result, mode) => {
    if (mode==="replace") { setProducts(result.products); setDiscrepancies(result.discrepancies); }
    else { setProducts(prev=>[...prev,...result.products]); setDiscrepancies(prev=>[...prev,...result.discrepancies]); }
    setSelectedProduct(null); setFilterCategory("all"); setSearch(""); setPage(1);
  }, []);

  const toggleSort = (col) => { if (sortBy===col) setSortDir(d=>-d); else { setSortBy(col); setSortDir(1); } setPage(1); };
  const sortIcon = (col) => sortBy===col ? (sortDir===1?" ↑":" ↓") : "";

  const detail = selectedProduct ? products.find(p=>p.id===selectedProduct) : null;
  const card = { background: "linear-gradient(135deg,#0f172a,#1e293b)", border: "1px solid #1e293b", borderRadius: 14, padding: 18 };
  const isEmpty = products.length===0;

  return (
    <div style={{ minHeight: "100vh", background: "#020617", color: "#e2e8f0", fontFamily: "'DM Sans', sans-serif" }}>
      <link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&family=JetBrains+Mono:wght@400;600&display=swap" rel="stylesheet" />
      {/* HEADER */}
      <div style={{ background: "linear-gradient(135deg,#0f172a,#1e1b4b 50%,#0f172a)", borderBottom: "1px solid #1e293b", padding: "14px 20px", display: "flex", alignItems: "center", justifyContent: "space-between", flexWrap: "wrap", gap: 10 }}>
        <div style={{ display: "flex", alignItems: "center", gap: 10 }}>
          <div style={{ width: 34, height: 34, borderRadius: 8, background: "linear-gradient(135deg,#2563eb,#7c3aed)", display: "flex", alignItems: "center", justifyContent: "center", fontSize: 16 }}>🛒</div>
          <h1 style={{ margin: 0, fontSize: 17, fontWeight: 800, background: "linear-gradient(90deg,#e2e8f0,#94a3b8)", WebkitBackgroundClip: "text", WebkitTextFillColor: "transparent" }}>SuperAnalytics</h1>
        </div>
        <div style={{ display: "flex", gap: 4, flexWrap: "wrap" }}>
          {["dashboard","discrepancias","kpi","inflacion"].map(v=>(
            <button key={v} onClick={()=>{setView(v);setSelectedProduct(null);setPage(1);}} style={{ padding: "6px 12px", borderRadius: 6, border: "none", cursor: "pointer", fontSize: 10, fontWeight: 600, background: view===v?"#2563eb":"#1e293b", color: view===v?"#fff":"#94a3b8" }}>
              {v==="dashboard"?"📊 Dashboard":v==="discrepancias"?"⚠️ Discrepancias":v==="kpi"?"📈 KPI":"📉 Inflación"}
            </button>
          ))}
          <button onClick={()=>setShowImport(true)} style={{ padding: "6px 12px", borderRadius: 6, border: "1px solid #059669", background: "rgba(5,150,105,.1)", color: "#34d399", cursor: "pointer", fontSize: 10, fontWeight: 600 }}>📥 Importar</button>
          {products.length>0&&<button onClick={()=>exportData(products)} style={{ padding: "6px 12px", borderRadius: 6, border: "1px solid #334155", background: "transparent", color: "#94a3b8", cursor: "pointer", fontSize: 10, fontWeight: 600 }}>📤 Exportar</button>}
        </div>
      </div>

      <div style={{ padding: "16px 20px", maxWidth: 1200, margin: "0 auto" }}>
        {isEmpty && (
          <div style={{ display: "flex", flexDirection: "column", alignItems: "center", paddingTop: 60, gap: 14, textAlign: "center" }}>
            <div style={{ fontSize: 50, opacity: .3 }}>📊</div>
            <h2 style={{ margin: 0, fontSize: 18, fontWeight: 700, color: "#64748b" }}>No hay datos cargados</h2>
            <p style={{ margin: 0, fontSize: 12, color: "#475569", maxWidth: 400 }}>Importá un Excel con Hoja 1 = Compras y Hoja 2 = Ventas.<br/>Columnas: F.Ingreso · Descripción artículo · Cantidad · Precio · Monto · Descripción</p>
            <div style={{ display: "flex", gap: 8, marginTop: 4 }}>
              <button onClick={()=>setShowImport(true)} style={{ padding: "10px 20px", borderRadius: 8, border: "none", background: "#2563eb", color: "#fff", cursor: "pointer", fontSize: 12, fontWeight: 600 }}>📥 Importar</button>
              <button onClick={downloadTemplate} style={{ padding: "10px 20px", borderRadius: 8, border: "1px solid #334155", background: "transparent", color: "#94a3b8", cursor: "pointer", fontSize: 12, fontWeight: 600 }}>📋 Plantilla</button>
            </div>
          </div>
        )}

        {!isEmpty && (
          <>
            <div style={{ display: "flex", gap: 8, marginBottom: 14, flexWrap: "wrap", alignItems: "center" }}>
              <SearchBar value={search} onChange={v=>{setSearch(v);setPage(1);}} />
              <label style={{ color: "#64748b", fontSize: 10, fontWeight: 600, marginLeft: 4 }}>CAT:</label>
              <select value={filterCategory} onChange={e=>{setFilterCategory(e.target.value);setPage(1);}} style={{ padding: "5px 8px", borderRadius: 5, background: "#1e293b", border: "1px solid #334155", color: "#e2e8f0", fontSize: 11 }}>
                {categories.map(c=><option key={c} value={c}>{c==="all"?"Todas":c}</option>)}
              </select>
              <span style={{ marginLeft: "auto", fontSize: 10, color: "#475569" }}>{filtered.length.toLocaleString()} artículos</span>
              {discrepancies.length>0&&<span style={{ fontSize: 10, color: "#fbbf24", fontWeight: 700, background: "rgba(251,191,36,.1)", padding: "2px 7px", borderRadius: 5 }}>⚠️ {discrepancies.length}</span>}
            </div>

            <InflationBanner />

            <div style={{ display: "grid", gridTemplateColumns: "repeat(auto-fit,minmax(140px,1fr))", gap: 8, marginBottom: 16 }}>
              {[
                {l:"Ingresos",v:fmt(stats.rev),c:"#4ade80",i:"💰"},{l:"Costos",v:fmt(stats.cost),c:"#f87171",i:"📦"},
                {l:"Ganancia",v:fmt(stats.profit),c:stats.profit>=0?"#4ade80":"#f87171",i:"🏆"},{l:"Margen",v:stats.margin.toFixed(1)+"%",c:"#a78bfa",i:"📐"},
                {l:"Comprado",v:stats.bought.toLocaleString()+" u.",c:"#38bdf8",i:"🔽"},{l:"Vendido",v:stats.sold.toLocaleString()+" u.",c:"#fbbf24",i:"🔼"},
              ].map(k=>(<div key={k.l} style={{...card,padding:12}}><div style={{display:"flex",justifyContent:"space-between"}}><span style={{color:"#64748b",fontSize:9,fontWeight:600,textTransform:"uppercase",letterSpacing:.8}}>{k.l}</span><span style={{fontSize:13}}>{k.i}</span></div><div style={{fontSize:18,fontWeight:800,color:k.c,fontFamily:"'JetBrains Mono',monospace",marginTop:2}}>{k.v}</div></div>))}
            </div>

            {/* DASHBOARD */}
            {view==="dashboard"&&!selectedProduct&&(<div>
              <Pagination total={filtered.length} page={page} perPage={PER_PAGE} onChange={setPage} />
              <div style={{ overflowX: "auto" }}>
                <table style={{ width: "100%", borderCollapse: "separate", borderSpacing: "0 4px", fontSize: 11 }}>
                  <thead><tr style={{ color: "#64748b", fontSize: 9, textTransform: "uppercase", letterSpacing: .8 }}>
                    <th onClick={()=>toggleSort("name")} style={{ textAlign: "left", padding: "6px 8px", cursor: "pointer" }}>Artículo{sortIcon("name")}</th>
                    <th style={{ textAlign: "left", padding: "6px 8px" }}>Cat.</th>
                    <th style={{ textAlign: "right", padding: "6px 8px" }}>Costo</th>
                    <th style={{ textAlign: "right", padding: "6px 8px" }}>Precio</th>
                    <th onClick={()=>toggleSort("margin")} style={{ textAlign: "right", padding: "6px 8px", cursor: "pointer" }}>Margen{sortIcon("margin")}</th>
                    <th onClick={()=>toggleSort("bought")} style={{ textAlign: "right", padding: "6px 8px", cursor: "pointer" }}>Compra{sortIcon("bought")}</th>
                    <th onClick={()=>toggleSort("sold")} style={{ textAlign: "right", padding: "6px 8px", cursor: "pointer" }}>Venta{sortIcon("sold")}</th>
                    <th onClick={()=>toggleSort("profit")} style={{ textAlign: "right", padding: "6px 8px", cursor: "pointer" }}>Ganancia{sortIcon("profit")}</th>
                    <th style={{ textAlign: "center", padding: "6px 8px" }}>Est.</th>
                  </tr></thead>
                  <tbody>{pageProducts.map(p=>{
                    const margin=p.avgPrice>0?((p.avgPrice-p.avgCost)/p.avgPrice)*100:0;
                    const profit=p.totalSaleAmount-p.totalCostAmount;
                    const issue=p.onlyPurchase||p.onlySale;
                    return (<tr key={p.id} onClick={()=>setSelectedProduct(p.id)} style={{ background: issue?"rgba(251,191,36,.04)":"#0f172a", cursor: "pointer" }} onMouseEnter={e=>e.currentTarget.style.background="#1e293b"} onMouseLeave={e=>e.currentTarget.style.background=issue?"rgba(251,191,36,.04)":"#0f172a"}>
                      <td style={{ padding: "8px", borderRadius: "6px 0 0 6px", fontWeight: 600, maxWidth: 200, overflow: "hidden", textOverflow: "ellipsis", whiteSpace: "nowrap" }}>{p.name}</td>
                      <td style={{ padding: 8 }}><span style={{ background: "#1e293b", padding: "2px 5px", borderRadius: 4, fontSize: 9, color: "#94a3b8" }}>{p.category}</span></td>
                      <td style={{ padding: 8, textAlign: "right", fontFamily: "'JetBrains Mono',monospace", color: "#f87171" }}>{fmt(p.avgCost)}</td>
                      <td style={{ padding: 8, textAlign: "right", fontFamily: "'JetBrains Mono',monospace", color: "#4ade80" }}>{fmt(p.avgPrice)}</td>
                      <td style={{ padding: 8, textAlign: "right", fontFamily: "'JetBrains Mono',monospace", color: margin>=25?"#4ade80":margin>=15?"#fbbf24":"#f87171", fontWeight: 700 }}>{margin.toFixed(1)}%</td>
                      <td style={{ padding: 8, textAlign: "right", fontFamily: "'JetBrains Mono',monospace", color: "#38bdf8" }}>{p.qtyBought.toLocaleString()}</td>
                      <td style={{ padding: 8, textAlign: "right", fontFamily: "'JetBrains Mono',monospace", color: "#fbbf24" }}>{p.qtySold.toLocaleString()}</td>
                      <td style={{ padding: 8, textAlign: "right", fontFamily: "'JetBrains Mono',monospace", color: profit>=0?"#4ade80":"#f87171", fontWeight: 700 }}>{fmt(profit)}</td>
                      <td style={{ padding: 8, textAlign: "center", borderRadius: "0 6px 6px 0" }}>
                        {p.onlyPurchase&&<span style={{ fontSize: 8, color: "#fb923c", fontWeight: 700, background: "rgba(251,146,60,.15)", padding: "1px 4px", borderRadius: 3 }}>COMPRA</span>}
                        {p.onlySale&&<span style={{ fontSize: 8, color: "#ef4444", fontWeight: 700, background: "rgba(239,68,68,.15)", padding: "1px 4px", borderRadius: 3 }}>VENTA</span>}
                        {!p.onlyPurchase&&!p.onlySale&&<span style={{ fontSize: 9, color: "#4ade80" }}>✓</span>}
                      </td>
                    </tr>);
                  })}</tbody>
                </table>
              </div>
              <Pagination total={filtered.length} page={page} perPage={PER_PAGE} onChange={setPage} />
            </div>)}

            {/* DETAIL */}
            {view==="dashboard"&&detail&&(<div style={{ display: "flex", flexDirection: "column", gap: 12 }}>
              <div style={{ display: "flex", alignItems: "center", gap: 8, flexWrap: "wrap" }}>
                <button onClick={()=>setSelectedProduct(null)} style={{ background: "#1e293b", border: "none", borderRadius: 5, color: "#94a3b8", cursor: "pointer", padding: "5px 10px", fontSize: 11 }}>← Volver</button>
                <h2 style={{ margin: 0, fontSize: 16, fontWeight: 700 }}>{detail.name}</h2>
                <span style={{ background: "#1e293b", padding: "2px 7px", borderRadius: 4, fontSize: 10, color: "#94a3b8" }}>{detail.category}</span>
              </div>
              <div style={{ display: "grid", gridTemplateColumns: "repeat(auto-fit,minmax(260px,1fr))", gap: 12 }}>
                <div style={card}>
                  <h3 style={{ margin: "0 0 10px", fontSize: 12, fontWeight: 700, color: "#94a3b8" }}>📊 Resumen</h3>
                  {[{l:"Cant. comprada",v:detail.qtyBought.toLocaleString()+" u.",c:"#38bdf8"},{l:"Cant. vendida",v:detail.qtySold.toLocaleString()+" u.",c:"#fbbf24"},
                    {l:"Costo promedio",v:fmt(detail.avgCost),c:"#f87171"},{l:"Precio promedio",v:fmt(detail.avgPrice),c:"#4ade80"},
                    {l:"Total compras",v:fmt(detail.totalCostAmount),c:"#f87171"},{l:"Total ventas",v:fmt(detail.totalSaleAmount),c:"#4ade80"},
                    {l:"Ganancia",v:fmt(detail.totalSaleAmount-detail.totalCostAmount),c:(detail.totalSaleAmount-detail.totalCostAmount)>=0?"#4ade80":"#f87171"},
                    {l:"Margen",v:(detail.avgPrice>0?((detail.avgPrice-detail.avgCost)/detail.avgPrice*100).toFixed(1):"0")+"%",c:"#a78bfa"},
                  ].map(r=>(<div key={r.l} style={{ display: "flex", justifyContent: "space-between", padding: "4px 0", borderBottom: "1px solid #1e293b" }}><span style={{ fontSize: 10, color: "#64748b" }}>{r.l}</span><span style={{ fontSize: 11, fontFamily: "'JetBrains Mono',monospace", color: r.c, fontWeight: 600 }}>{r.v}</span></div>))}
                </div>
                <div style={card}>
                  <h3 style={{ margin: "0 0 10px", fontSize: 12, fontWeight: 700, color: "#94a3b8" }}>📦 Compra vs Venta</h3>
                  {(()=>{ const mx=Math.max(detail.qtyBought,detail.qtySold); return (<div style={{ display: "flex", flexDirection: "column", gap: 8 }}>
                    <div><div style={{ display: "flex", justifyContent: "space-between", marginBottom: 3 }}><span style={{ fontSize: 10, color: "#38bdf8" }}>Comprado</span><span style={{ fontSize: 10, fontFamily: "'JetBrains Mono',monospace", color: "#38bdf8" }}>{detail.qtyBought.toLocaleString()}</span></div><MiniBar value={detail.qtyBought} max={mx} color="#38bdf8" /></div>
                    <div><div style={{ display: "flex", justifyContent: "space-between", marginBottom: 3 }}><span style={{ fontSize: 10, color: "#fbbf24" }}>Vendido</span><span style={{ fontSize: 10, fontFamily: "'JetBrains Mono',monospace", color: "#fbbf24" }}>{detail.qtySold.toLocaleString()}</span></div><MiniBar value={detail.qtySold} max={mx} color="#fbbf24" /></div>
                    <div style={{ fontSize: 10, color: "#64748b" }}>Dif: <span style={{ color: detail.qtyBought>=detail.qtySold?"#4ade80":"#f87171", fontWeight: 700 }}>{(detail.qtyBought-detail.qtySold).toLocaleString()} u.</span></div>
                  </div>); })()}
                </div>
                <div style={card}><h3 style={{ margin: "0 0 10px", fontSize: 12, fontWeight: 700, color: "#94a3b8" }}>📈 Cambios de Precio</h3><KPIChanges product={detail} /></div>
              </div>
            </div>)}

            {view==="discrepancias"&&(<div style={{ display: "flex", flexDirection: "column", gap: 12 }}>
              <h2 style={{ margin: 0, fontSize: 14, fontWeight: 700, color: "#94a3b8" }}>⚠️ Discrepancias Compras vs Ventas</h2>
              <p style={{ margin: 0, fontSize: 11, color: "#64748b" }}>Artículos que están en una hoja pero no en la otra. Sirve para detectar errores de registro.</p>
              <DiscrepancyTable discrepancies={discrepancies} />
            </div>)}

            {view==="kpi"&&(<div style={{ display: "flex", flexDirection: "column", gap: 12 }}>
              <h2 style={{ margin: 0, fontSize: 14, fontWeight: 700, color: "#94a3b8" }}>📈 KPI — Variaciones</h2>
              <Pagination total={filtered.length} page={page} perPage={PER_PAGE} onChange={setPage} />
              {pageProducts.map(p=>{
                const fC=p.costHistory[0]?.cost||0, lC=p.costHistory[p.costHistory.length-1]?.cost||0;
                const fP=p.priceHistory[0]?.price||0, lP=p.priceHistory[p.priceHistory.length-1]?.price||0;
                const cC=fC>0?((lC-fC)/fC)*100:0, pC=fP>0?((lP-fP)/fP)*100:0;
                return (<div key={p.id} style={card}>
                  <div style={{ display: "flex", justifyContent: "space-between", marginBottom: 8, flexWrap: "wrap", gap: 4 }}>
                    <span style={{ fontWeight: 700, fontSize: 13 }}>{p.name} <span style={{ background: "#1e293b", padding: "2px 5px", borderRadius: 4, fontSize: 9, color: "#94a3b8" }}>{p.category}</span></span>
                    <div style={{ display: "flex", gap: 12 }}>
                      <div style={{ textAlign: "right" }}><div style={{ fontSize: 8, color: "#64748b" }}>COSTO</div><div style={{ fontFamily: "'JetBrains Mono',monospace", fontSize: 12, fontWeight: 700, color: cC>0?"#f87171":"#4ade80" }}>{pct(cC)}</div></div>
                      <div style={{ textAlign: "right" }}><div style={{ fontSize: 8, color: "#64748b" }}>PRECIO</div><div style={{ fontFamily: "'JetBrains Mono',monospace", fontSize: 12, fontWeight: 700, color: pC>0?"#4ade80":"#f87171" }}>{pct(pC)}</div></div>
                    </div>
                  </div>
                  <KPIChanges product={p} />
                </div>);
              })}
              <Pagination total={filtered.length} page={page} perPage={PER_PAGE} onChange={setPage} />
            </div>)}

            {view==="inflacion"&&(<div style={{ display: "flex", flexDirection: "column", gap: 12 }}>
              <h2 style={{ margin: 0, fontSize: 14, fontWeight: 700, color: "#94a3b8" }}>📉 Inflación y Estimación</h2>
              <InflationBanner />
              <Pagination total={filtered.length} page={page} perPage={PER_PAGE} onChange={setPage} />
              <InflationEstimator products={filtered} page={page} perPage={PER_PAGE} />
            </div>)}
          </>
        )}
      </div>
      {showImport&&<ImportPanel onImport={handleImport} onClose={()=>setShowImport(false)} count={products.length} />}
    </div>
  );
}